In [ ]:
!pip install selenium -q
!pip install webdriver-manager -q
!apt-get update -q
!apt-get install -y chromium-browser chromium-chromedriver -q
!pip install google_colab_selenium -q# если вы будете пользоваться коллабом

In [ ]:
from selenium import webdriver
import pandas as pd
from tqdm import tqdm
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from bs4 import BeautifulSoup
import time
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.chrome.service import Service

# Запуск браузера
# Настройки WebDriver
# Если запускаете локально

chrome_options = webdriver.ChromeOptions()
chrome_options.add_argument("--blink-settings=imagesEnabled=false")
chrome_options.add_argument("headless")
chrome_options.add_argument("no-sandbox")
chrome_options.add_argument("disable-dev-shm-usage")
driver = webdriver.Chrome(options=chrome_options)


# Запуск браузера
# Настройки WebDriver
# Если запускаете в коллаб

import google_colab_selenium as gs
driver = gs.Chrome()


def scroll_down():
    """Функция для прокрутки страницы вниз"""
    driver.find_element(By.TAG_NAME, "body").send_keys(Keys.END)
    time.sleep(2)  # Ждем загрузки новых элементов

# query1 Karelia все новости
# query2 Карелия арктические зоны
# query3 Карелия развитие

url1 = 'https://www.rbc.ru/search/?query=Карелия'
url2 = 'https://www.rbc.ru/search/?query=%D0%9A%D0%B0%D1%80%D0%B5%D0%BB%D0%B8%D1%8F%20%D0%B0%D1%80%D0%BA%D1%82%D0%B8%D1%87%D0%B5%D1%81%D0%BA%D0%B8%D0%B5%20%D0%B7%D0%BE%D0%BD%D1%8B'
url3 = 'https://www.rbc.ru/search/?query=%D0%9A%D0%B0%D1%80%D0%B5%D0%BB%D0%B8%D1%8F%20%D1%80%D0%B0%D0%B7%D0%B2%D0%B8%D1%82%D0%B8%D0%B5'

try:
    url = url3 # Заменить на нужный URL
    driver.get(url)
    time.sleep(3)  # Даем странице загрузиться

    seen_links = set()
    articles = []
    last_height = driver.execute_script("return document.body.scrollHeight")

    for _ in tqdm(range(3)):  # Прокручиваем страницу
        scroll_down()
        new_height = driver.execute_script("return document.body.scrollHeight")
        if new_height == last_height:
            break  # Если высота не изменилась, значит, долистали до конца
        last_height = new_height

    soup = BeautifulSoup(driver.page_source, "html.parser")

    for item in soup.select(".search-item"):
        link_tag = item.select_one(".search-item__link")
        if not link_tag:
            continue

        link = link_tag["href"]
        if link in seen_links:
            continue  # Пропускаем дубликаты
        seen_links.add(link)

        title_tag = link_tag.select_one(".search-item__title")
        title = title_tag.text.strip() if title_tag else "Без заголовка"

        text_tag = link_tag.select_one(".search-item__text")
        short_text = text_tag.text.strip() if text_tag else ""

        time_tag = item.select_one(".search-item__category")
        time_source = time_tag.text.strip() if time_tag else "Без времени"

        articles.append({
            "title": title,
            "short_text": short_text,
            "time_source": time_source,
            "link": link
        })

    # # Парсим полные тексты статей
    # for article in articles:
    #     driver.get(article["link"])
    #     time.sleep(2)

    #     article_soup = BeautifulSoup(driver.page_source, "html.parser")
    #     paragraphs = article_soup.select("p")
    #     full_text = " ".join(p.text.strip() for p in paragraphs if p.text.strip())

    #     article["full_text"] = full_text

    # # Вывод результатов
    # for article in articles:
    #     print(f"Заголовок: {article['title']}")
    #     print(f"Источник и время: {article['time_source']}")
    #     print(f"Ссылка: {article['link']}")
    #     print(f"Краткое описание: {article['short_text']}")
    #  #   print(f"Полный текст: {article['full_text'][:500]}...")  # Ограничение на 500 символов
    #     print("=" * 80)

finally:
    driver.quit()

<IPython.core.display.Javascript object>

100%|██████████| 3/3 [00:06<00:00,  2.12s/it]


In [ ]:
karelia_query_data = pd.DataFrame(articles)
karelia_query_data

,title,short_text,time_source,link
0,Фермеры Карелии получили более 19 млн руб. на ...,,"РБК,\n 13:00",https://karelia.rbc.ru/karelia/05/12/2025/6932...
1,В Карелии на развитие малого агробизнеса напра...,,"РБК,\n 03 дек, 15:26",https://karelia.rbc.ru/karelia/03/12/2025/6930...
2,Глава Карелии открыл новую площадку литейного ...,На предприятии запустили новый участок предвар...,"РБК+,\n Пресс-релиз...",https://karelia.plus.rbc.ru/pressrelease/69303...
3,В Карелии назначили исполняющего обязанности г...,,"РБК,\n 02 дек, 13:26",https://karelia.rbc.ru/karelia/02/12/2025/692e...
4,Власти в Карелии требуют гарантийного ремонта ...,,"РБК,\n 01 дек, 16:05",https://karelia.rbc.ru/karelia/01/12/2025/692d...
...,...,...,...,...
75,Наталья Зубаревич: о гонке на стероидах в росс...,Что происходит с российской экономикой в 2025 ...,"Основной, 29 авг, 09:00",https://pro.rbc.ru/demo/68b030ae9a7947ed2c1b7d3d
76,В Костомукше создадут новый рыбоводный комплек...,,"РБК,\n 28 авг, 18:00",https://karelia.rbc.ru/karelia/28/08/2025/68b0...
77,Карелии одобрен инфраструктурный кредит на мод...,Глава Карелии Артур Парфенчиков обсудил с заме...,"РБК+,\n Пресс-релиз...",https://karelia.plus.rbc.ru/pressrelease/68b05...
78,Прокуратура утвердила обвинительное заключение...,,"РБК,\n 27 авг, 16:10",https://karelia.rbc.ru/karelia/27/08/2025/68af...


In [ ]:
karelia_query_data.duplicated().sum()